In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [2]:
from pathlib import Path
ROOT = _here if (_here / "data").exists() else _here.parent
DATA, RESULTS = ROOT / "data", ROOT / "results"

In [ ]:
"""
H1 Analysis: License Declaration vs. Provenance Source Reuse
--------------------------------------------------------------
H1. Repositories without an explicit license declaration are more frequently reused as a provenance source than repositories with one.

Design notes:
- `has_license` is derived from `source_file_license`, a repository-level declaration rather than a per-file parse, and is
  aggregated to a per-repository signal via LICENSE_AGGREGATION.
- Repository age (`age_days`) is built from `source_version`, which carries genuine per-commit variation (1,134,414 of 1,181,642 source
  repositories show more than one distinct value).
- `sink_version` is NOT used as a covariate: it is predominantly a project-level scan-time constant (7,715 of 10,337 projects have
  exactly one distinct value), not a per-method timestamp.
- Repository popularity was unavailable and is not included.

Reproduces: N = 17,368 repositories (11,314 licensed / 6,054 unlicensed); Cliff's delta = 0.030; age-controlled Negative Binomial
beta = 0.014, p = .332 (log-age: beta = 0.012, p = .384).
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

# =============================================================================
# CONFIGURATION
# =============================================================================

RELATIONSHIPS_CSV = DATA / "license_analysis_results_processed.csv"

EXCLUDE_CATEGORY_0 = True   # violation_lcd_category == 0 -> "no match", source/sink swapped
EXCLUDE_NA_SINK = True      # sink_repository_url starts with "N/A"

REUSE_COUNT_MODE = "distinct_sink_projects"  # or "distinct_relationships"

# How to aggregate potentially inconsistent file-level licenses into a
# single per-repository has_license signal:
#   "any_licensed"      -> True if ANY observed file license is a real declaration
#   "majority_licensed" -> True if >50% of observed file licenses are real declarations
LICENSE_AGGREGATION = "any_licensed"

GITHUB_URL_PATTERN = r'(https://github\.com/[^/]+/[^/]+)/.*'

RUN_INTERACTIVE_DIAGNOSTICS = True


# =============================================================================
# Stage 1: Load, deduplicate, diagnose
# =============================================================================

def load_relationships(path):
    df = pd.read_csv(path)
    before = len(df)
    df = df.drop_duplicates(subset=["method_hash", "source_repository_url", "sink_repository_url"])
    print(f"[dedup] {before} raw rows -> {len(df)} after dedup "
          f"({before - len(df)} duplicates removed, {(before - len(df)) / before:.3%})")
    return df


def run_diagnostics(df):
    print("\n=== project_type distribution ===")
    print(df["project_type"].value_counts(dropna=False))

    print("\n=== violation_lcd_category distribution ===")
    print(df["violation_lcd_category"].value_counts(dropna=False).sort_index())

    na_mask = df["sink_repository_url"].astype(str).str.startswith("N/A")
    print(f"\n=== 'N/A' sink URLs: {na_mask.sum()} rows ({na_mask.sum() / len(df):.3%}) ===")
    print(df.loc[na_mask, "violation_lcd_category"].value_counts(dropna=False).sort_index())

    null_mask = df["sink_repository_url"].isna()
    print(f"\n=== null sink_repository_url: {null_mask.sum()} rows ===")
    print(df.loc[null_mask, "violation_lcd_category"].value_counts(dropna=False).sort_index())

    print("\n=== source_file_license: distinct raw values (top 20) ===")
    print(df["source_file_license"].value_counts(dropna=False).head(20))

    print("\n=== source_version: null count ===")
    print(f"{df['source_version'].isna().sum()} / {len(df)} rows have null source_version")


# =============================================================================
# Stage 2: Filter
# =============================================================================

def filter_relationships(df):
    out = df.copy()
    n0 = len(out)

    if EXCLUDE_CATEGORY_0:
        out = out[out["violation_lcd_category"] != 0]
    out = out[out["sink_repository_url"].notna()]
    if EXCLUDE_NA_SINK:
        out = out[~out["sink_repository_url"].astype(str).str.startswith("N/A")]

    print(f"[filter] {n0} -> {len(out)} rows after category-0 / null / N/A exclusions "
          f"({n0 - len(out)} removed, {(n0 - len(out)) / n0:.3%})")
    return out


def extract_repo_root(url_series):
    extracted = url_series.astype(str).str.extract(GITHUB_URL_PATTERN, expand=False)
    n_failed = extracted.isna().sum()
    if n_failed > 0:
        print(f"[warn] {n_failed} URLs did not match the expected github.com/org/repo "
              f"pattern and will be dropped ({n_failed / len(url_series):.3%})")
    return extracted


# =============================================================================
# Stage 3: Build per-source-repository table
#   (reuse_count + has_license + age_days)
# =============================================================================

def normalize_license_declaration(license_value):
    if pd.isna(license_value):
        return False

    val_clean = str(license_value).strip().lower()

    no_declaration_values = {
        "none", "noassertion", "unlicensed", "-", "n/a", "", "other",
        "<undefined>", "undefined", "null", "unknown"
    }

    return val_clean not in no_declaration_values


def build_source_repo_table(df):
    """
    Builds one row per unique source repository with:
      - reuse_count : per REUSE_COUNT_MODE
      - has_license : aggregated from source_file_license across all observed occurrences of that repo as a source
      - age_days    : derived from the earliest observed source_version (commit timestamp, epoch ms) for that repo,
                       relative to the most recent timestamp observed anywhere in the filtered dataset
    """
    df = df.copy()
    df["source_repo_root"] = extract_repo_root(df["source_repository_url"])
    df = df.dropna(subset=["source_repo_root"])
    df["is_licensed_row"] = df["source_file_license"].apply(normalize_license_declaration)

    # --- reuse_count ---
    if REUSE_COUNT_MODE == "distinct_sink_projects":
        df["sink_repo_root"] = extract_repo_root(df["sink_repository_url"])
        df_valid_sink = df.dropna(subset=["sink_repo_root"])
        reuse_count = df_valid_sink.groupby("source_repo_root")["sink_repo_root"].nunique()
    elif REUSE_COUNT_MODE == "distinct_relationships":
        reuse_count = df.groupby("source_repo_root").size()
    else:
        raise ValueError(f"Unknown REUSE_COUNT_MODE: {REUSE_COUNT_MODE}")

    # --- has_license aggregation ---
    if LICENSE_AGGREGATION == "any_licensed":
        has_license = df.groupby("source_repo_root")["is_licensed_row"].any()
    elif LICENSE_AGGREGATION == "majority_licensed":
        has_license = df.groupby("source_repo_root")["is_licensed_row"].mean() > 0.5
    else:
        raise ValueError(f"Unknown LICENSE_AGGREGATION: {LICENSE_AGGREGATION}")

    # --- age_days (NEW) ---
    reference_ts = df["source_version"].max()
    earliest_ts = df.groupby("source_repo_root")["source_version"].min()
    age_days = (reference_ts - earliest_ts) / (1000 * 60 * 60 * 24)

    out = pd.DataFrame({
        "reuse_count": reuse_count,
        "has_license": has_license,
        "age_days": age_days,
    }).reset_index().rename(columns={"source_repo_root": "repo_url"})
    out["reuse_count"] = out["reuse_count"].fillna(0).astype(int)
    out["has_license"] = out["has_license"].fillna(False)

    n_missing_age = out["age_days"].isna().sum()
    if n_missing_age > 0:
        print(f"[warn] {n_missing_age} / {len(out)} repos have missing age_days "
              f"(no valid source_version observed) "
              f"({n_missing_age/len(out):.2%})")

    # Diagnostic: how often is license inconsistent across observed files
    # for the same repo?
    consistency = df.groupby("source_repo_root")["is_licensed_row"].agg(["mean", "count"])
    inconsistent = consistency[(consistency["mean"] > 0) & (consistency["mean"] < 1) & (consistency["count"] > 1)]
    print(f"\n[license consistency] {len(inconsistent)} / {len(consistency)} source repos "
          f"({len(inconsistent)/len(consistency):.2%}) show mixed licensed/unlicensed "
          f"file observations across rows.")

    return out


# =============================================================================
# Stage 4: Statistical analysis
# =============================================================================

def check_overdispersion(df):
    mean_ = df["reuse_count"].mean()
    var_ = df["reuse_count"].var()
    ratio = var_ / mean_ if mean_ > 0 else np.nan
    print(f"\nreuse_count: mean={mean_:.3f}, variance={var_:.3f}, ratio={ratio:.2f}")
    print("  -> Negative Binomial appropriate" if ratio > 1.5
          else "  -> Little overdispersion; compare to Poisson via AIC")


def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    greater = (x[:, None] > y[None, :]).sum()
    less = (x[:, None] < y[None, :]).sum()
    return (greater - less) / (len(x) * len(y))


def mannwhitney_h1(df):
    """Primary analysis, matching Table I's stated methodology (unchanged)."""
    licensed = df.loc[df["has_license"], "reuse_count"]
    unlicensed = df.loc[~df["has_license"], "reuse_count"]

    stat, p = stats.mannwhitneyu(unlicensed, licensed, alternative="two-sided")
    delta = cliffs_delta(unlicensed.values, licensed.values)

    print(f"\nN licensed = {len(licensed)}, N unlicensed = {len(unlicensed)}")
    print(f"Median reuse_count -- licensed: {licensed.median():.1f}, "
          f"unlicensed: {unlicensed.median():.1f}")
    print(f"Mann-Whitney U = {stat:.1f}, p = {p:.4g}")
    print(f"Cliff's delta = {delta:.4f}  "
          f"({'unlicensed > licensed' if delta > 0 else 'licensed > unlicensed' if delta < 0 else 'no difference'})")

    return {
        "U": stat, "p_value": p, "cliffs_delta": delta,
        "n_licensed": len(licensed), "n_unlicensed": len(unlicensed),
        "median_licensed": licensed.median(), "median_unlicensed": unlicensed.median(),
    }


def run_negbin_regression(df):
    """
    Original single-predictor model (has_license only), retained for
    direct comparison against the age-controlled version below.
    """
    y = df["reuse_count"]
    X = sm.add_constant(df["has_license"].astype(int))

    model = sm.NegativeBinomial(y, X).fit(disp=False)
    print(model.summary())

    summary = pd.DataFrame({
        "coef": model.params,
        "IRR": np.exp(model.params),
        "IRR_CI_low": np.exp(model.conf_int()[0]),
        "IRR_CI_high": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    })
    return model, summary


def run_negbin_regression_with_age(df):
    """
    Extended model including age_days as a covariate. age_days is standardized (z-scored) before fitting -- the raw day-scale value
    (in the thousands) causes numerical overflow in the MLE optimizer (exp(linpred) overflow, non-convergence, singular Hessian). This
    is a standard fix for scale-related MLE instability, not a change in what's being modeled: the standardized coefficient is
    interpreted as "effect per standard deviation of age," and the IRR is reported per SD accordingly.
    """
    before = len(df)
    df_complete = df.dropna(subset=["age_days"]).copy()
    dropped = before - len(df_complete)
    print(f"\n[age model] {before} -> {len(df_complete)} repos with valid age_days "
          f"({dropped} dropped, {dropped/before:.2%})")

    age_mean = df_complete["age_days"].mean()
    age_std = df_complete["age_days"].std()
    df_complete["age_years"] = df_complete["age_days"] / 365.25
    df_complete["age_std"] = (df_complete["age_days"] - age_mean) / age_std

    print(f"[age model] age_days: mean={age_mean:.1f}, std={age_std:.1f}, "
          f"min={df_complete['age_days'].min():.1f}, max={df_complete['age_days'].max():.1f}")

    y = df_complete["reuse_count"]
    X = sm.add_constant(pd.DataFrame({
        "has_license": df_complete["has_license"].astype(int),
        "age_std": df_complete["age_std"],
    }))

    model = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=200)
    print(model.summary())

    if not model.mle_retvals.get("converged", True):
        print("[warn] Model still did not converge -- inspect further before trusting results.")

    summary = pd.DataFrame({
        "coef": model.params,
        "IRR": np.exp(model.params),
        "IRR_CI_low": np.exp(model.conf_int()[0]),
        "IRR_CI_high": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    })
    return model, summary


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    raw = load_relationships(RELATIONSHIPS_CSV)
    run_diagnostics(raw)

    if RUN_INTERACTIVE_DIAGNOSTICS:
        input("\n>>> Review diagnostics above (esp. source_file_license values -- "
              "confirm normalize_license_declaration()'s no-declaration set covers "
              "everything seen). Press Enter to continue...")

    filtered = filter_relationships(raw)
    df = build_source_repo_table(filtered)

    print(f"\nN = {len(df)} unique source repositories "
          f"({df['has_license'].sum()} licensed, {(~df['has_license']).sum()} unlicensed)")
    print(f"reuse_count mode: {REUSE_COUNT_MODE}, license aggregation: {LICENSE_AGGREGATION}")

    check_overdispersion(df)

    print("\n--- Primary analysis: Mann-Whitney U + Cliff's delta ---")
    mw_results = mannwhitney_h1(df)

    print("\n--- Supplementary A: Negative Binomial regression (has_license only) ---")
    model_base, nb_summary_base = run_negbin_regression(df)

    print("\n--- Supplementary B: Negative Binomial regression (has_license + age_days) ---")
    model_age, nb_summary_age = run_negbin_regression_with_age(df)

    df.to_csv(RESULTS / "h1_analysis_dataframe.csv", index=False)
    nb_summary_base.to_csv(RESULTS / "h1_negbin_summary_base.csv")
    nb_summary_age.to_csv(RESULTS / "h1_negbin_summary_with_age.csv")
    pd.DataFrame([mw_results]).to_csv(RESULTS / "h1_mannwhitney_summary.csv", index=False)

    print("\n[done] Results written to h1_analysis_dataframe.csv, "
          "h1_negbin_summary_base.csv, h1_negbin_summary_with_age.csv, "
          "h1_mannwhitney_summary.csv")

[dedup] 1183182 raw rows -> 1183182 after dedup (0 duplicates removed, 0.000%)

=== project_type distribution ===
project_type
1    543616
3    330640
2    308926
Name: count, dtype: int64

=== violation_lcd_category distribution ===
violation_lcd_category
0    314813
1    181631
2     62880
3    130193
4     17948
5    475717
Name: count, dtype: int64

=== 'N/A' sink URLs: 264355 rows (22.343%) ===
violation_lcd_category
5    264355
Name: count, dtype: int64

=== null sink_repository_url: 319413 rows ===
violation_lcd_category
0    314813
1      3275
2       548
3       129
4        14
5       634
Name: count, dtype: int64

=== source_file_license: distinct raw values (top 20) ===
source_file_license
Other                                             313804
Apache License 2.0                                245366
MIT License                                       140598
NaN                                               135896
GNU General Public License v2.0                    87658
-   

In [ ]:
"""
H1 Sanity Check: Age Distribution Diagnostics and Log-Age Robustness
------------------------------------------------------------------------
Investigates:
  1. Whether age_days == 0 repos are a genuine artifact (single observation coinciding with the dataset's max timestamp) vs. a
     real, if unusual, case.
  2. Whether the age_std effect in the NegBin model is robust to a log-transform, given the right-skewed distribution
     (mean ~2292 days, std ~1995 days).

Assumes `df` (the filtered relationship-level data) and `out` (the per-source-repo table from build_source_repo_table) are
available, or rebuilds them from the pipeline if run standalone.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm

# =============================================================================
# Part 1: Investigate age_days == 0 repos
# =============================================================================

def diagnose_zero_age(df_filtered, repo_table):
    """
    df_filtered: the row-level filtered dataframe (post filter_relationships)
    repo_table:  the per-repo table from build_source_repo_table (has age_days)
    """
    zero_age_repos = repo_table[repo_table["age_days"] == 0]["repo_url"]
    print(f"=== Repos with age_days == 0: {len(zero_age_repos)} / {len(repo_table)} "
          f"({len(zero_age_repos)/len(repo_table):.2%}) ===")

    if len(zero_age_repos) == 0:
        print("No zero-age repos found.")
        return

    # For each zero-age repo, how many distinct source_version values
    # did it actually have, and how many rows contributed?
    df_filtered = df_filtered.copy()
    df_filtered["source_repo_root"] = df_filtered["source_repository_url"].astype(str).str.extract(
        r'(https://github\.com/[^/]+/[^/]+)/.*', expand=False
    )
    zero_age_rows = df_filtered[df_filtered["source_repo_root"].isin(zero_age_repos)]

    per_repo_stats = zero_age_rows.groupby("source_repo_root").agg(
        n_rows=("source_version", "size"),
        n_distinct_versions=("source_version", "nunique"),
        n_null_versions=("source_version", lambda s: s.isna().sum()),
    )
    print("\n--- Row/version counts for zero-age repos (sample) ---")
    print(per_repo_stats.describe())
    print(f"\nRepos with exactly 1 observed row: "
          f"{(per_repo_stats['n_rows'] == 1).sum()} / {len(per_repo_stats)}")
    print(f"Repos with exactly 1 distinct version: "
          f"{(per_repo_stats['n_distinct_versions'] == 1).sum()} / {len(per_repo_stats)}")

    reference_ts = df_filtered["source_version"].max()
    print(f"\nDataset reference (max) timestamp: {reference_ts} "
          f"({pd.to_datetime(reference_ts, unit='ms')})")

    return per_repo_stats


# =============================================================================
# Part 2: Full age distribution shape
# =============================================================================

def describe_age_distribution(repo_table):
    age = repo_table["age_days"].dropna()
    print("\n=== age_days distribution ===")
    print(age.describe())
    print(f"\nSkewness: {age.skew():.3f}")
    print(f"Percentiles: 1%={age.quantile(0.01):.1f}, 5%={age.quantile(0.05):.1f}, "
          f"25%={age.quantile(0.25):.1f}, 50%={age.quantile(0.50):.1f}, "
          f"75%={age.quantile(0.75):.1f}, 95%={age.quantile(0.95):.1f}, "
          f"99%={age.quantile(0.99):.1f}")
    print(f"\nCount at age_days == 0: {(age == 0).sum()}")
    print(f"Count at age_days < 1 day: {(age < 1).sum()}")
    print(f"Count at age_days < 7 days: {(age < 7).sum()}")


# =============================================================================
# Part 3: Log-age robustness model
# =============================================================================

def run_negbin_regression_log_age(repo_table):
    """
    Robustness check: log(age_days + 1) instead of standardized raw
    age_days, to test whether the strong age effect and the
    has_license non-effect are robust to the skewed distribution.
    +1 offset handles age_days == 0 cases without dropping them.
    """
    df_complete = repo_table.dropna(subset=["age_days"]).copy()
    df_complete["log_age"] = np.log1p(df_complete["age_days"])  # log(age_days + 1)

    print(f"\n[log-age model] N = {len(df_complete)}")
    print(f"log_age: mean={df_complete['log_age'].mean():.3f}, "
          f"std={df_complete['log_age'].std():.3f}, "
          f"skew={df_complete['log_age'].skew():.3f}")

    y = df_complete["reuse_count"]
    X = sm.add_constant(pd.DataFrame({
        "has_license": df_complete["has_license"].astype(int),
        "log_age": df_complete["log_age"],
    }))

    model = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=200)
    print(model.summary())

    if not model.mle_retvals.get("converged", True):
        print("[warn] Log-age model did not converge.")

    summary = pd.DataFrame({
        "coef": model.params,
        "IRR": np.exp(model.params),
        "IRR_CI_low": np.exp(model.conf_int()[0]),
        "IRR_CI_high": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    })
    return model, summary


# =============================================================================
# Main (assumes `filtered` and `df` [the repo table] already exist from
# the main H1 script; otherwise rebuild them here)
# =============================================================================

if __name__ == "__main__":
    # If running standalone, uncomment and adjust:
    # raw = load_relationships(RELATIONSHIPS_CSV)
    # filtered = filter_relationships(raw)
    # df = build_source_repo_table(filtered)

    describe_age_distribution(df)
    diagnose_zero_age(filtered, df)

    print("\n" + "=" * 60)
    print("LOG-AGE ROBUSTNESS MODEL")
    print("=" * 60)
    model_log, summary_log = run_negbin_regression_log_age(df)

    summary_log.to_csv(RESULTS / "h1_negbin_summary_log_age.csv")
    print("\n[done] Log-age results written to h1_negbin_summary_log_age.csv")


=== age_days distribution ===
count    17368.000000
mean      2292.235557
std       1994.940146
min          0.000000
25%        937.515966
50%       1806.478640
75%       3231.486641
max      20538.529097
Name: age_days, dtype: float64

Skewness: 3.423
Percentiles: 1%=217.5, 5%=334.5, 25%=937.5, 50%=1806.5, 75%=3231.5, 95%=5689.7, 99%=7500.2

Count at age_days == 0: 1
Count at age_days < 1 day: 1
Count at age_days < 7 days: 2
=== Repos with age_days == 0: 1 / 17368 (0.01%) ===

--- Row/version counts for zero-age repos (sample) ---
       n_rows  n_distinct_versions  n_null_versions
count     1.0                  1.0              1.0
mean      1.0                  1.0              0.0
std       NaN                  NaN              NaN
min       1.0                  1.0              0.0
25%       1.0                  1.0              0.0
50%       1.0                  1.0              0.0
75%       1.0                  1.0              0.0
max       1.0                  1.0          

In [ ]:
"""
H1a Analysis: License Family and Provenance Source Reuse
------------------------------------------------------------------------
H1a. Controlling for repository age, methods originating from permissively licensed repositories are reused more frequently than
methods originating from reciprocally (copyleft) licensed repositories.

Design notes:
- License family is classified by exact dict lookup through DSR_engine.py (license_mapping -> get_license_group). Substring
  matching is not viable: source_file_license stores full spelled-out names such as "GNU General Public License v2.0", which contain no
  short abbreviation to match on. H3 uses this identical implementation, so the two hypotheses' classifications cannot drift
  apart. Cross-validated against an independent implementation across all 1,183,182 rows with 0 disagreements (classifier_crossvalidation.ipynb).
- Repositories with an Unresolved license family are excluded. The hypothesis concerns the Permissive-vs-Reciprocal contrast, not
  declared-vs-undeclared, which is H1's question.
- The age covariate (age_days) uses the same source_version-derived construction as H1.

Because reuse_count is >= 1 by construction, the estimates here are checked against a support-corrected specification in h1a_truncation_robustness.ipynb.

Reproduces: N = 11,252 repositories (9,035 Permissive / 2,217 Reciprocal) after excluding 6,116 Unresolved; IRR = 1.047,
p = .029 (Holm-adjusted p = .059); log-age: IRR = 1.020, p = .340.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm

GITHUB_URL_PATTERN = r'(https://github\.com/[^/]+/[^/]+)/.*'

# ---- License classification: DSR_engine.py is the single source of truth ----
_ns = {}
exec(open(DATA / "DSR_engine.py").read(), _ns)  # adjust path if DSR_engine.py lives elsewhere
license_mapping = _ns["license_mapping"]
get_license_group = _ns["get_license_group"]


def classify_license(lic):
    """Returns 'Permissive', 'Reciprocal', or 'Unresolved'.

    Exact dict lookup: raw license string -> SPDX id (via license_mapping)
    -> compatibility group (via get_license_group) -> Permissive /
    Weak+Strong Copyleft (folded into 'Reciprocal') / anything else
    (Custom, Restricted, Proprietary, Unknown, Unlicensed -> 'Unresolved').

    No substring matching, so no risk of false-positive keyword collisions
    (e.g. the earlier "upl" matching inside "eupl" bug). Raw strings not
    present in license_mapping at all also resolve to Unresolved -- this
    is a deliberate conservative default, not a silent failure.
    """
    if pd.isna(lic):
        return "Unresolved"
    spdx = license_mapping.get(str(lic).strip(), None)
    if spdx is None:
        return "Unresolved"
    group = get_license_group(spdx)
    if group == "Permissive":
        return "Permissive"
    if group in ("Weak Copyleft", "Strong Copyleft"):
        return "Reciprocal"
    return "Unresolved"

"""
def extract_repo_root(url_series):
    return url_series.astype(str).str.extract(GITHUB_URL_PATTERN, expand=False)
"""

def build_h1a_table(df):
    """
    One row per unique source repository with:
      - reuse_count       : distinct sink projects reusing this repo
      - license_family    : Permissive / Reciprocal (Unresolved excluded)
      - age_days          : same construction as H1
    """
    df = df.copy()
    df["source_repo_root"] = extract_repo_root(df["source_repository_url"])
    df = df.dropna(subset=["source_repo_root"])
    df["license_family"] = df["source_file_license"].apply(classify_license)

    df["sink_repo_root"] = extract_repo_root(df["sink_repository_url"])
    df_valid_sink = df.dropna(subset=["sink_repo_root"])
    reuse_count = df_valid_sink.groupby("source_repo_root")["sink_repo_root"].nunique()

    # majority license family per repo (mode of observed rows)
    license_family = df.groupby("source_repo_root")["license_family"].agg(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "Unresolved"
    )

    reference_ts = df["source_version"].max()
    earliest_ts = df.groupby("source_repo_root")["source_version"].min()
    age_days = (reference_ts - earliest_ts) / (1000 * 60 * 60 * 24)

    out = pd.DataFrame({
        "reuse_count": reuse_count,
        "license_family": license_family,
        "age_days": age_days,
    }).reset_index().rename(columns={"source_repo_root": "repo_url"})
    out["reuse_count"] = out["reuse_count"].fillna(0).astype(int)

    before = len(out)
    out = out[out["license_family"].isin(["Permissive", "Reciprocal"])].copy()
    print(f"[H1a] {before} -> {len(out)} repos after excluding Unresolved license family "
          f"({before - len(out)} removed)")

    out = out.dropna(subset=["age_days"])
    out["is_permissive"] = (out["license_family"] == "Permissive").astype(int)
    out["age_std"] = (out["age_days"] - out["age_days"].mean()) / out["age_days"].std()

    return out


def run_h1a_regression(df):
    y = df["reuse_count"]
    X = sm.add_constant(pd.DataFrame({
        "is_permissive": df["is_permissive"],
        "age_std": df["age_std"],
    }))
    model = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=200)
    print(model.summary())

    summary = pd.DataFrame({
        "coef": model.params,
        "IRR": np.exp(model.params),
        "IRR_CI_low": np.exp(model.conf_int()[0]),
        "IRR_CI_high": np.exp(model.conf_int()[1]),
        "p_value": model.pvalues,
    })
    return model, summary


if __name__ == "__main__":
    df = pd.read_csv(DATA / "license_analysis_results_processed.csv")
    df = df.drop_duplicates(subset=["method_hash", "source_repository_url", "sink_repository_url"])
    df = df[df["violation_lcd_category"] != 0]
    df = df[df["sink_repository_url"].notna()]
    df = df[~df["sink_repository_url"].astype(str).str.startswith("N/A")]

    h1a_table = build_h1a_table(df)
    h1a_table.to_csv(RESULTS / "h1a_analysis_dataframe.csv", index=False)
    print(f"\nN = {len(h1a_table)} repos "
          f"({(h1a_table['license_family']=='Permissive').sum()} Permissive, "
          f"{(h1a_table['license_family']=='Reciprocal').sum()} Reciprocal)")

    model, summary = run_h1a_regression(h1a_table)
    summary.to_csv(RESULTS / "h1a_negbin_summary.csv")

[warn] 23512 URLs did not match the expected github.com/org/repo pattern and will be dropped (3.922%)
[H1a] 17368 -> 11252 repos after excluding Unresolved license family (6116 removed)

N = 11252 repos (9035 Permissive, 2217 Reciprocal)
                     NegativeBinomial Regression Results                      
Dep. Variable:            reuse_count   No. Observations:                11252
Model:               NegativeBinomial   Df Residuals:                    11249
Method:                           MLE   Df Model:                            2
Date:                Mon, 24 Aug 2026   Pseudo R-squ.:                 0.02262
Time:                        19:03:41   Log-Likelihood:                -19340.
converged:                       True   LL-Null:                       -19788.
Covariance Type:            nonrobust   LLR p-value:                4.206e-195
                    coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------

In [6]:
# Check for outliers within the Reciprocal group specifically
reciprocal = h1a_table[h1a_table["license_family"] == "Reciprocal"]
print(reciprocal["reuse_count"].describe())
print(reciprocal.nlargest(10, "reuse_count")[["repo_url", "reuse_count", "age_days"]])

count    2217.000000
mean        2.004060
std         2.530051
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        46.000000
Name: reuse_count, dtype: float64
                                          repo_url  reuse_count     age_days
14266       https://github.com/raphael/linux-samus           46  3936.849618
14333           https://github.com/reactos/reactos           31  8180.666123
2979   https://github.com/OpenLiberty/open-liberty           29  3112.504433
8260          https://github.com/geotools/geotools           29  5497.603646
13283               https://github.com/openjdk/jfx           26  4703.407882
6340       https://github.com/chenshuo/linux-debug           23  3652.955579
11321     https://github.com/metasfresh/metasfresh           17  3798.255880
14753          https://github.com/samba-team/samba           17  8785.955023
16015            https://github.com/tutao/tutanota           17  3818.666331
16583           https://gith

In [7]:
import numpy as np

h1a_table["log_age"] = np.log1p(h1a_table["age_days"])

print(f"log_age: mean={h1a_table['log_age'].mean():.3f}, "
      f"std={h1a_table['log_age'].std():.3f}, "
      f"skew={h1a_table['log_age'].skew():.3f}")

import statsmodels.api as sm

y = h1a_table["reuse_count"]
X = sm.add_constant(pd.DataFrame({
    "is_permissive": h1a_table["is_permissive"],
    "log_age": h1a_table["log_age"],
}))

model_log = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=200)
print(model_log.summary())

summary_log = pd.DataFrame({
    "coef": model_log.params,
    "IRR": np.exp(model_log.params),
    "IRR_CI_low": np.exp(model_log.conf_int()[0]),
    "IRR_CI_high": np.exp(model_log.conf_int()[1]),
    "p_value": model_log.pvalues,
})
print(summary_log)

log_age: mean=7.318, std=0.842, skew=-0.286
                     NegativeBinomial Regression Results                      
Dep. Variable:            reuse_count   No. Observations:                11252
Model:               NegativeBinomial   Df Residuals:                    11249
Method:                           MLE   Df Model:                            2
Date:                Mon, 24 Aug 2026   Pseudo R-squ.:                 0.02535
Time:                        19:03:42   Log-Likelihood:                -19286.
converged:                       True   LL-Null:                       -19788.
Covariance Type:            nonrobust   LLR p-value:                1.436e-218
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -1.7493      0.080    -21.786      0.000      -1.907      -1.592
is_permissive     0.0199      0.021      0.954      0.340      -0.021       0.